In [ ]:
import ee
import geemap 
# ee.Authenticate()
# ee.Initialize()
from importlib import reload  

Initialize GEE project

In [ ]:
geemap.ee_initialize(project="water-sinapohlabeln")

Import modules

In [ ]:
from modules import ms_indices_CO2 as indices
from modules import configs, utils_string
from modules import utils_Landsat_SR_CO2 as utils_LS
from modules import high_level_functions_CO2
from modules import utils_geom

Reload modules

In [ ]:
utils_string = reload(utils_string)
reload(indices)
reload(utils_LS)
reload(high_level_functions_CO2)

Set Parameters

In [ ]:
# PROPERTIES
# SET METADATA PARAMETERS
MAXCLOUD = 70 
STARTYEAR = 2005
ENDYEAR = 2024
STARTMONTH = 7
ENDMONTH = 8
SCALE = 30

SIZE_LON = 10
SIZE_LAT = 5
longitudes = range(-160, -140, SIZE_LON)
latitudes = range(65, 75, SIZE_LAT)
# longitudes = [-135]
# latitudes = [68, 69]
BUFFER_SIZE = 0.001

target_collection = 'users/ingmarnitze/TCTrend_SR_2005-2024_TCVIS'
#target_collection = 'projects/water-sinapohlabeln/assets/TCTrend_SR_2005-2024_TCVIS'

Image metadata

In [ ]:
# image metadata Filters
config_trend = {
  'STARTYEAR': STARTYEAR,
  'ENDYEAR': ENDYEAR,
  'max_cloud_cover': MAXCLOUD,
  'date_filter_yr' : ee.Filter.calendarRange(STARTYEAR, ENDYEAR, 'year'),
  'date_filter_mth' : ee.Filter.calendarRange(STARTMONTH, ENDMONTH, 'month'),
  'meta_filter_cld' : ee.Filter.lt('CLOUD_COVER', MAXCLOUD),
  'select_bands_visible' : ["SR_B1", "SR_B2","SR_B3","SR_B4"],
  'select_indices' : ["TCB", "TCG", "TCW"],
  'select_TCtrend_bands' : ["TCB_slope", "TCG_slope", "TCW_slope"],
  'geom' : None,
  'longitudes' : longitudes,
  'latitudes' : latitudes
}
#------ RUN FULL PROCESS FOR ALL REGIONS IN LOOP ------------------------------

In [ ]:
RUN = 0
m = geemap.Map()

Run & export trends

In [ ]:
for lowLat in latitudes:
    for leftLon in longitudes:
        
        
        # check for Hemisphere
        if lowLat < 0:
            sizeLat = SIZE_LAT * -1
        else:
            sizeLat = SIZE_LAT
            
        sizeLon = SIZE_LON

        geom = utils_geom.create_buffered_rectangle(leftLon, lowLat, sizeLon=sizeLon, sizeLat=sizeLat, buffer_size=BUFFER_SIZE, geodesic=False)

        config_trend['geom'] = geom 
        m.addLayer(geom,{}, str(lowLat))

        #File name
        assetname_new = utils_string.make_TCTrendAssetNameSR(leftLon, lowLat, STARTYEAR, ENDYEAR)
        assetId = target_collection + '/' + assetname_new + 'test_510'

        # Calculate Trend
        trend = high_level_functions_CO2.runTCTrend(config_trend)
        if RUN:
            task = ee.batch.Export.image.toAsset(
                image=ee.Image(trend['visual']).toByte(),
                description=assetname_new,
                assetId=assetId,
                scale=SCALE,
                region=geom,
                maxPixels=1e12)

            task.start()



Visualize Geometry 

In [ ]:
m